# Notebook 05 — Comparative Analysis
## Reads ONLY saved artifacts — zero model loading required

**Prerequisite:** All four member notebooks (01–04) must be fully executed first.

This notebook never imports a model, never calls inference, and never reads video files.
It consumes only the JSON/CSV artifacts saved to `results/` and produces all
paper-level analysis, tables, and figures centrally.

| Section | Content |
|---------|----------|
| §1 | Artifact inventory & completeness check |
| §2 | Aggregate results tables (MSVD · MSR-VTT) |
| §3 | Cross-dataset generalisation analysis |
| §4 | Statistical significance tests (Wilcoxon) |
| §5 | Per-video score distributions & error analysis |
| §6 | Inference latency comparison |
| §7 | All ablation studies (frames · VTM · quantisation) |
| §8 | Qualitative analysis & failure mode taxonomy |
| §9 | Vid2Seq dense caption analysis |
| §10 | Paper-ready LaTeX tables & master publication figure |


---
## §1 — Artifact Inventory & Completeness Check

In [ ]:
import os, json, glob, warnings, textwrap, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from collections import defaultdict, Counter
from itertools import combinations
warnings.filterwarnings('ignore')

with open('shared/config.json') as f: cfg = json.load(f)
RESULTS_DIR = cfg['RESULTS_DIR']
FIGS_DIR    = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR  = os.path.join(RESULTS_DIR, 'tables')
for d in [FIGS_DIR, TABLES_DIR]: os.makedirs(d, exist_ok=True)

METRICS  = ['BLEU_1','BLEU_4','METEOR','CIDEr','ROUGE_L']
DATASETS = ['MSVD','MSR-VTT']
PALETTE  = ['#4C72B0','#55A868','#C44E52','#8172B2',
            '#CCB974','#64B5CD','#E47A2E']

MODEL_LABELS = {
    'CLIP_plus_GPT_2' :'CLIP+GPT-2',
    'ViViT_plus_GPT_2':'ViViT+GPT-2',
    'BLIP_2'          :'BLIP-2',
    'CLIP4Caption'    :'CLIP4Caption',
    'LLaVA_NeXT'      :'LLaVA-NeXT',
    'Video_LLaMA_2'   :'Video-LLaMA-2',
    'Vid2Seq'         :'Vid2Seq',
}
MODELS = list(MODEL_LABELS.keys())
print(f'Results dir: {RESULTS_DIR}')
print(f'Models: {list(MODEL_LABELS.values())}')

In [ ]:
# Expected artifact suffixes per model per dataset
EXPECTED = ['_scores.json','_hypotheses.json','_references.json',
            '_examples.csv','_per_video_scores.csv','_timing.json']
status = {}
missing = []
for safe, label in MODEL_LABELS.items():
    for ds in DATASETS:
        tag = f'{safe}_{ds}'
        row = {}
        for suf in EXPECTED:
            p = os.path.join(RESULTS_DIR, f'{tag}{suf}')
            row[suf] = os.path.exists(p)
            if not row[suf]: missing.append(f'{tag}{suf}')
        status[f'{label} / {ds}'] = row
df_status = pd.DataFrame(status).T
df_status.columns = [s.replace('_','') for s in EXPECTED]
print('Artifact completeness:')
print(df_status.to_string())
if missing:
    print(f'\nWARNING: {len(missing)} missing files:')
    for m in missing: print(f'  {m}')
else:
    print('\n All expected artifacts present.')

---
## §2 — Aggregate Results Tables

In [ ]:
scores = defaultdict(dict)
for safe, label in MODEL_LABELS.items():
    for ds in DATASETS:
        p = os.path.join(RESULTS_DIR, f'{safe}_{ds}_scores.json')
        if os.path.exists(p):
            with open(p) as f: d = json.load(f)
            scores[ds][label] = d['scores']
        else:
            scores[ds][label] = {m: 0.0 for m in METRICS}

dfs = {}
for ds in DATASETS:
    df = pd.DataFrame(scores[ds]).T[METRICS]
    df.index.name = 'Model'
    dfs[ds] = df
    print(f'\n{"="*64}')
    print(f'  {ds} Test Set')
    print(f'{"="*64}')
    print(df.to_string())
    best = df.idxmax()
    print('  Best: ' + '  '.join(f'{m}:{best[m]}' for m in METRICS))
    df.to_csv(os.path.join(RESULTS_DIR, f'scores_{ds}.csv'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
for ax, ds in zip(axes, DATASETS):
    df  = dfs[ds]
    mdl = list(df.index)
    x   = np.arange(len(METRICS))
    n   = len(mdl); w = 0.10
    for i, (model, color) in enumerate(zip(mdl, PALETTE)):
        vals   = [df.loc[model, m] for m in METRICS]
        offset = (i - n/2 + 0.5) * w
        bars   = ax.bar(x + offset, vals, w, label=model,
                        color=color, alpha=0.88, edgecolor='white', linewidth=0.4)
        for b, v in zip(bars, vals):
            if v > 0.005:
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                        f'{v:.2f}', ha='center', va='bottom',
                        fontsize=5.5, rotation=45)
    ax.set_title(f'{ds} Test Set', fontsize=13)
    ax.set_xticks(x); ax.set_xticklabels(METRICS, fontsize=10)
    ax.set_ylabel('Score', fontsize=10)
    ax.legend(fontsize=7.5, ncol=2)
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.grid(True, alpha=0.3, linestyle='--'); ax.set_axisbelow(True)
fig.suptitle('All Models — MSVD & MSR-VTT Comparative Results', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR,'fig_main_results.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_main_results.png'), dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig_main_results.pdf / .png')

---
## §3 — Cross-Dataset Generalisation Analysis

In [ ]:
common = [m for m in dfs['MSVD'].index if m in dfs['MSR-VTT'].index]
df_delta = (dfs['MSR-VTT'].loc[common] - dfs['MSVD'].loc[common]).round(4)
df_delta.index.name = 'Model'
print('Cross-dataset delta (MSR-VTT minus MSVD)  negative = harder on MSR-VTT')
print('='*64)
print(df_delta.to_string())
df_delta.to_csv(os.path.join(RESULTS_DIR,'cross_dataset_delta.csv'))
print('Saved cross_dataset_delta.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
ax = axes[0]
data_h = df_delta.values.astype(float)
im = ax.imshow(data_h, cmap='RdYlGn', aspect='auto', vmin=-0.15, vmax=0.15)
plt.colorbar(im, ax=ax, label='Score delta')
ax.set_xticks(range(len(METRICS))); ax.set_xticklabels(METRICS, fontsize=9)
ax.set_yticks(range(len(df_delta))); ax.set_yticklabels(df_delta.index, fontsize=9)
ax.set_title('Generalisation Heatmap (MSR-VTT minus MSVD)', fontsize=11)
for i in range(len(df_delta)):
    for j in range(len(METRICS)):
        ax.text(j,i,f'{data_h[i,j]:+.3f}',ha='center',va='center',fontsize=8)
ax = axes[1]
xp = np.arange(len(common))
b1 = ax.bar(xp-0.2,[dfs['MSVD'].loc[m,'CIDEr'] for m in common],
            0.38, label='MSVD', color='#4C72B0', alpha=0.85)
b2 = ax.bar(xp+0.2,[dfs['MSR-VTT'].loc[m,'CIDEr'] for m in common],
            0.38, label='MSR-VTT', color='#C44E52', alpha=0.85)
for bars in [b1,b2]:
    for b in bars:
        v=b.get_height()
        if v>0.005: ax.text(b.get_x()+b.get_width()/2, v+0.003,
                            f'{v:.3f}',ha='center',va='bottom',fontsize=7.5)
ax.set_title('CIDEr: MSVD vs MSR-VTT', fontsize=11)
ax.set_xticks(xp); ax.set_xticklabels(common, rotation=22, ha='right', fontsize=8)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3, linestyle='--'); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR,'fig_cross_dataset.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_cross_dataset.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## §4 — Statistical Significance Tests

Wilcoxon signed-rank test (non-parametric, paired) on per-video CIDEr scores. α = 0.05.

In [ ]:
pv_scores = defaultdict(dict)
for safe, label in MODEL_LABELS.items():
    for ds in DATASETS:
        p = os.path.join(RESULTS_DIR, f'{safe}_{ds}_per_video_scores.csv')
        if os.path.exists(p):
            pv_scores[ds][label] = pd.read_csv(p).set_index('video_id')
print('Per-video score files loaded:')
for ds in DATASETS:
    print(f'  {ds}: {list(pv_scores[ds].keys())}')

In [ ]:
def wilcoxon_matrix(ds_label, metric='CIDEr'):
    avail = list(pv_scores[ds_label].keys())
    n = len(avail)
    pmat = pd.DataFrame(np.ones((n,n)), index=avail, columns=avail)
    for m1, m2 in combinations(avail, 2):
        s1 = pv_scores[ds_label][m1]
        s2 = pv_scores[ds_label][m2]
        common_v = s1.index.intersection(s2.index)
        if len(common_v) < 10: continue
        a = s1.loc[common_v, metric].values.astype(float)
        b = s2.loc[common_v, metric].values.astype(float)
        if np.allclose(a, b): continue
        _, p = stats.wilcoxon(a, b, zero_method='pratt')
        pmat.loc[m1,m2] = round(p,4)
        pmat.loc[m2,m1] = round(p,4)
    return pmat

for ds in DATASETS:
    if not pv_scores[ds]: print(f'No per-video data for {ds}'); continue
    pm = wilcoxon_matrix(ds, 'CIDEr')
    print(f'\nWilcoxon p-value matrix (CIDEr) — {ds}:')
    print(pm.to_string())
    sig = [(m1,m2) for m1,m2 in combinations(pm.index,2) if pm.loc[m1,m2]<0.05]
    print(f'  Significant pairs (p<0.05): {len(sig)}')
    for m1,m2 in sig: print(f'    {m1} vs {m2}  p={pm.loc[m1,m2]}')
    pm.to_csv(os.path.join(RESULTS_DIR,f'wilcoxon_cider_{ds}.csv'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, ds in zip(axes, DATASETS):
    if not pv_scores[ds]: ax.axis('off'); continue
    pm   = wilcoxon_matrix(ds, 'CIDEr')
    mdls = list(pm.index)
    im   = ax.imshow(pm.values.astype(float), cmap='RdYlGn_r', vmin=0, vmax=0.1)
    plt.colorbar(im, ax=ax, label='p-value')
    ax.set_xticks(range(len(mdls)))
    ax.set_xticklabels(mdls, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(mdls)))
    ax.set_yticklabels(mdls, fontsize=8)
    ax.set_title(f'Wilcoxon p-values (CIDEr) — {ds}\nred = significant', fontsize=11)
    for i in range(len(mdls)):
        for j in range(len(mdls)):
            v = pm.values[i,j]
            txt = chr(8212) if i==j else f'{v:.3f}'
            ax.text(j,i,txt,ha='center',va='center',fontsize=7,
                    color='white' if v<0.03 else 'black')
plt.suptitle('Statistical Significance — Wilcoxon Signed-Rank', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR,'fig_wilcoxon.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_wilcoxon.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## §5 — Per-Video Score Distributions & Error Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, ds in zip(axes, DATASETS):
    if not pv_scores[ds]: ax.axis('off'); continue
    mdls  = [m for m in MODEL_LABELS.values() if m in pv_scores[ds]]
    data  = [pv_scores[ds][m]['CIDEr'].values.astype(float) for m in mdls]
    parts = ax.violinplot(data, positions=range(len(mdls)),
                          showmedians=True, showextrema=True)
    for pc, color in zip(parts['bodies'], PALETTE):
        pc.set_facecolor(color); pc.set_alpha(0.7)
    ax.set_xticks(range(len(mdls)))
    ax.set_xticklabels(mdls, rotation=25, ha='right', fontsize=8)
    ax.set_ylabel('Per-video CIDEr', fontsize=10)
    ax.set_title(f'CIDEr Distribution — {ds}', fontsize=11)
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.grid(True, alpha=0.3, linestyle='--'); ax.set_axisbelow(True)
plt.suptitle('Per-Video CIDEr Distributions', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR,'fig_per_video_violin.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_per_video_violin.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Hardest videos: lowest max CIDEr across all models
for ds in DATASETS:
    if not pv_scores[ds]: continue
    mdls = list(pv_scores[ds].keys())
    all_vids = set()
    for m in mdls: all_vids.update(pv_scores[ds][m].index.tolist())
    max_cider = {}
    for vid in all_vids:
        vals = [pv_scores[ds][m].loc[vid,'CIDEr']
                for m in mdls if vid in pv_scores[ds][m].index]
        max_cider[vid] = max(vals) if vals else 0.0
    hard = sorted(max_cider, key=max_cider.get)[:20]
    df_hard = pd.DataFrame([{'video_id':v,'max_CIDEr_any_model':round(max_cider[v],4)}
                             for v in hard])
    p = os.path.join(RESULTS_DIR, f'error_hard_videos_{ds}.csv')
    df_hard.to_csv(p, index=False)
    print(f'\n{ds} — 20 hardest videos:')
    print(df_hard.to_string(index=False))
    print(f'Saved {os.path.basename(p)}')

In [ ]:
# Failure mode taxonomy
def classify_failure(hyp, ref):
    h = hyp.lower().strip(); r = ref.lower().strip()
    if not h or h in ('a video clip','a video','the video'): return 'empty/generic'
    h_w = set(h.split()); r_w = set(r.split())
    overlap = len(h_w & r_w) / max(len(r_w),1)
    if overlap > 0.6: return 'correct'
    action_words = {'running','walking','playing','cooking','dancing',
                    'swimming','riding','jumping','talking','sitting'}
    if not h_w & action_words and r_w & action_words: return 'missing action'
    if len(h.split()) < 4: return 'too short'
    if overlap < 0.1: return 'hallucination'
    return 'partial match'

hyps = defaultdict(dict); refs = defaultdict(dict)
for safe, label in MODEL_LABELS.items():
    for ds in DATASETS:
        ph = os.path.join(RESULTS_DIR, f'{safe}_{ds}_hypotheses.json')
        pr = os.path.join(RESULTS_DIR, f'{safe}_{ds}_references.json')
        if os.path.exists(ph):
            with open(ph) as f: hyps[ds][label] = json.load(f)
        if os.path.exists(pr):
            with open(pr) as f: refs[ds][label] = json.load(f)

for ds in DATASETS:
    avail = [m for m in MODEL_LABELS.values() if hyps[ds].get(m) and refs[ds].get(m)]
    if not avail: continue
    counts = {m: Counter() for m in avail}
    common_v = set(hyps[ds][avail[0]].keys())
    for m in avail[1:]: common_v &= set(hyps[ds][m].keys())
    for vid in common_v:
        ref0 = refs[ds][avail[0]].get(vid,[''])
        ref0 = ref0[0] if isinstance(ref0,list) else ref0
        for m in avail:
            h = hyps[ds][m].get(vid,'')
            counts[m][classify_failure(h, ref0)] += 1
    df_fail = pd.DataFrame(counts).T.fillna(0).astype(int)
    df_fail.index.name = 'Model'
    print(f'\nFailure taxonomy — {ds}:')
    print(df_fail.to_string())
    fp = os.path.join(RESULTS_DIR, f'failure_taxonomy_{ds}.csv')
    df_fail.to_csv(fp)
    print(f'Saved {os.path.basename(fp)}')

---
## §6 — Inference Latency Comparison

In [ ]:
timing = defaultdict(dict)
for safe, label in MODEL_LABELS.items():
    for ds in DATASETS:
        p = os.path.join(RESULTS_DIR, f'{safe}_{ds}_timing.json')
        if os.path.exists(p):
            with open(p) as f: timing[ds][label] = json.load(f)

rows = []
for label in MODEL_LABELS.values():
    row = {'Model': label}
    for ds in DATASETS:
        if label in timing[ds]:
            row[f'{ds} mean(s)'] = timing[ds][label]['mean_s']
            row[f'{ds} std(s)']  = timing[ds][label]['std_s']
    rows.append(row)
df_timing = pd.DataFrame(rows).set_index('Model')
print('Latency summary (seconds/video):'); print(df_timing.to_string())
df_timing.to_csv(os.path.join(RESULTS_DIR,'latency_summary.csv'))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
mdls_lt = list(MODEL_LABELS.values())
xp = np.arange(len(mdls_lt)); w = 0.35
for j, ds in enumerate(DATASETS):
    means = [timing[ds].get(m,{}).get('mean_s',0) for m in mdls_lt]
    stds  = [timing[ds].get(m,{}).get('std_s',0)  for m in mdls_lt]
    color = '#4C72B0' if ds=='MSVD' else '#C44E52'
    bars  = ax.bar(xp+(j-0.5)*w, means, w, label=ds, color=color,
                   alpha=0.85, yerr=stds, error_kw={'elinewidth':1.2,'capsize':4})
    for b,v in zip(bars,means):
        if v>0: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                        f'{v:.2f}s',ha='center',va='bottom',fontsize=8)
ax.set_xticks(xp); ax.set_xticklabels(mdls_lt, rotation=22, ha='right', fontsize=9)
ax.set_ylabel('Inference time (s/video)', fontsize=10)
ax.set_title('Mean Inference Latency per Video (plus/minus 1 std)', fontsize=12)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3, linestyle='--'); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR,'fig_latency.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_latency.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## §7 — All Ablation Studies

In [ ]:
abl_frame = defaultdict(dict)
abl_vtm   = {}
abl_quant = {}
for safe, label in MODEL_LABELS.items():
    for ds in DATASETS:
        p = os.path.join(RESULTS_DIR, f'{safe}_{ds}_ablation_frames.json')
        if os.path.exists(p):
            with open(p) as f: d = json.load(f)
            abl_frame[label][ds] = d['results']
p_vtm = os.path.join(RESULTS_DIR,'CLIP4Caption_vtm_ablation.json')
if os.path.exists(p_vtm):
    with open(p_vtm) as f: abl_vtm = json.load(f)
p_qt = os.path.join(RESULTS_DIR,'LLaVA_NeXT_quantisation_ablation.json')
if os.path.exists(p_qt):
    with open(p_qt) as f: abl_quant = json.load(f)
abl_models = [m for m in MODEL_LABELS.values() if abl_frame.get(m)]
print(f'Frame ablation models: {abl_models}')
print(f'VTM ablation: {list(abl_vtm.keys())}')
print(f'Quant ablation: {list(abl_quant.keys())}')

In [ ]:
FRAME_COUNTS = [4, 8, 16]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for row_i, metric in enumerate(['CIDEr','BLEU_4']):
    for col_j, ds in enumerate(DATASETS):
        ax = axes[row_i][col_j]
        for model, color in zip(abl_models, PALETTE):
            if ds not in abl_frame.get(model,{}): continue
            abl = abl_frame[model][ds]
            vals = [abl.get(f'{n} frames',{}).get('scores',{}).get(metric,0)
                    for n in FRAME_COUNTS]
            ax.plot(FRAME_COUNTS, vals, 'o-', label=model,
                    color=color, linewidth=2, markersize=7)
            for x_,y_ in zip(FRAME_COUNTS,vals):
                ax.annotate(f'{y_:.3f}',(x_,y_),
                            textcoords='offset points',xytext=(0,8),
                            ha='center',fontsize=8)
        ax.set_title(f'{metric} vs Frame Count — {ds}', fontsize=11)
        ax.set_xlabel('Frames sampled'); ax.set_ylabel(metric)
        ax.set_xticks(FRAME_COUNTS); ax.legend(fontsize=8)
        ax.spines[['top','right']].set_visible(False)
        ax.yaxis.grid(True,alpha=0.3,linestyle='--'); ax.set_axisbelow(True)
plt.suptitle('Frame-Count Ablation: All Models x Both Datasets', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR,'fig_ablation_frames.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_ablation_frames.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
if abl_vtm:
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    for ax, ds in zip(axes, DATASETS):
        if ds not in abl_vtm: ax.axis('off'); continue
        variants = list(abl_vtm[ds].keys())
        xp = np.arange(len(variants))
        for j, metric in enumerate(['CIDEr','BLEU_4']):
            vals = [abl_vtm[ds][v].get('scores',abl_vtm[ds][v]).get(metric,0)
                    for v in variants]
            color = '#8172B2' if metric=='CIDEr' else '#CCB974'
            bars = ax.bar(xp+(j-0.5)*0.35, vals, 0.33, label=metric,
                          color=color, alpha=0.85)
            for b,v in zip(bars,vals):
                ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.002,
                        f'{v:.3f}',ha='center',va='bottom',fontsize=8.5)
        ax.set_title(f'VTM Re-Ranking Ablation — {ds}',fontsize=11)
        ax.set_xticks(xp); ax.set_xticklabels(variants,fontsize=9)
        ax.legend(fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        ax.yaxis.grid(True,alpha=0.3,linestyle='--'); ax.set_axisbelow(True)
    plt.suptitle('CLIP4Caption: VTM Re-Ranking Ablation', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS_DIR,'fig_ablation_vtm.pdf'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(FIGS_DIR,'fig_ablation_vtm.png'), dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('VTM ablation not found.')

In [ ]:
if abl_quant:
    rows_q = []
    for qlabel, d in abl_quant.items():
        s = d.get('scores', d)
        t = d.get('timing',{})
        rows_q.append({'Quantisation': qlabel,
                        **{m: s.get(m,0) for m in METRICS},
                        'mean_latency_s': t.get('mean_s', float('nan'))})
    df_quant = pd.DataFrame(rows_q).set_index('Quantisation')
    print('LLaVA-NeXT Quantisation Ablation (MSVD 100-video subset):')
    print(df_quant.to_string())
    df_quant.to_csv(os.path.join(RESULTS_DIR,'quantisation_ablation_summary.csv'))
    print('Saved quantisation_ablation_summary.csv')
else:
    print('Quantisation ablation not found.')

---
## §8 — Qualitative Analysis

In [ ]:
wrap = lambda s,w=52: '\n                  '.join(textwrap.wrap(str(s),w))
for ds in DATASETS:
    avail = [m for m in MODEL_LABELS.values() if hyps[ds].get(m)]
    if not avail: continue
    all_vids = list(hyps[ds][avail[0]].keys())
    n_qual   = min(8, len(all_vids))
    sel_vids = [all_vids[i] for i in
                np.linspace(0,len(all_vids)-1,n_qual,dtype=int)]
    rows = []
    print(f'\n{"="*76}\nQualitative — {ds}\n{"="*76}')
    for vid in sel_vids:
        ref_list = refs[ds][avail[0]].get(vid,[''])
        ref_list = ref_list if isinstance(ref_list,list) else [ref_list]
        print(f'\nVideo: {vid}')
        print(f'  Ref[0]: {wrap(ref_list[0])}')
        if len(ref_list)>1: print(f'  Ref[1]: {wrap(ref_list[1])}')
        row = {'video_id':vid,'ref_0':ref_list[0],
               'ref_1':ref_list[1] if len(ref_list)>1 else ''}
        for m in avail:
            h = hyps[ds][m].get(vid,'-')
            print(f'  {m:<18}: {wrap(h)}')
            row[m] = h
        rows.append(row)
    qp = os.path.join(RESULTS_DIR, f'qualitative_8vids_{ds}.csv')
    pd.DataFrame(rows).to_csv(qp, index=False)
    print(f'Saved {os.path.basename(qp)}')

---
## §9 — Vid2Seq Dense Caption Analysis

In [ ]:
dense_caps = {}; time_dists = {}
for ds in DATASETS:
    dp = os.path.join(RESULTS_DIR, f'Vid2Seq_dense_captions_{ds}.json')
    tp = os.path.join(RESULTS_DIR, f'Vid2Seq_time_token_dist_{ds}.json')
    if os.path.exists(dp):
        with open(dp) as f: dense_caps[ds] = json.load(f)
    if os.path.exists(tp):
        with open(tp) as f: time_dists[ds] = json.load(f)['distribution']
for ds,d in dense_caps.items():
    n_ev = sum(len(v) for v in d.values())
    n_w  = sum(1 for v in d.values() if v)
    print(f'{ds}: {len(d)} videos | {n_w} with events | {n_ev} total | {n_ev/max(n_w,1):.1f} avg/video')

In [ ]:
if time_dists:
    fig, axes = plt.subplots(1, len(time_dists), figsize=(14,4))
    if len(time_dists)==1: axes=[axes]
    for ax, (ds, dist) in zip(axes, time_dists.items()):
        bins = sorted(int(k) for k in dist.keys())
        vals = [dist[str(b)] for b in bins]
        ax.bar(bins, vals, color='#E47A2E', alpha=0.85, edgecolor='white', linewidth=0.3)
        wm = np.average(bins, weights=vals)
        ax.axvline(wm, color='#C44E52', linewidth=2, linestyle='--',
                   label=f'Weighted mean={wm:.1f}')
        ax.set_xlabel('Time token (0=start, 99=end)', fontsize=10)
        ax.set_ylabel('Event count', fontsize=10)
        ax.set_title(f'Vid2Seq Time-Token Distribution — {ds}', fontsize=11)
        ax.legend(fontsize=9)
        ax.spines[['top','right']].set_visible(False)
        ax.yaxis.grid(True,alpha=0.3,linestyle='--'); ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS_DIR,'fig_vid2seq_time_tokens.pdf'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(FIGS_DIR,'fig_vid2seq_time_tokens.png'), dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
for ds, d in dense_caps.items():
    vids = list(d.keys())[:8]
    print(f'\nDense captions — {ds} (8 videos):')
    print('='*68)
    for vid in vids:
        events = d[vid]
        print(f'\nVideo: {vid}')
        if events:
            for e in events: print(f'  [{e["pct"]:.0f}%] {e["caption"]}')
        else: print('  (no events detected)')

---
## §10 — Paper-Ready LaTeX Tables & Master Publication Figure

In [ ]:
def to_latex(df, caption, label, highlight_best=True):
    cols = list(df.columns)
    best_idx = df.idxmax() if highlight_best else {}
    def fmt(val, col, idx):
        try: v = float(val)
        except: return str(val)
        s = f'{v:.4f}'
        if highlight_best and best_idx.get(col)==idx:
            s = f'\\textbf{{{s}}}'
        return s
    colspec = 'l' + 'c'*len(cols)
    lines = ['\\begin{table}[t]','\\centering',
             f'\\caption{{{caption}}}',f'\\label{{{label}}}',
             f'\\begin{{tabular}}{{{colspec}}}','\\toprule',
             'Model & ' + ' & '.join(cols) + ' \\\\','\\midrule']
    for idx, row in df.iterrows():
        cells = [str(idx)] + [fmt(row[c],c,idx) for c in cols]
        lines.append(' & '.join(cells) + ' \\\\')
    lines += ['\\bottomrule','\\end{tabular}','\\end{table}']
    return '\n'.join(lines)

table_defs = [
    ('tab_msvd',   dfs['MSVD'],  'Comparative results on MSVD test set. Bold=best.','tab:msvd'),
    ('tab_msrvtt', dfs['MSR-VTT'],'Comparative results on MSR-VTT test set. Bold=best.','tab:msrvtt'),
    ('tab_delta',  df_delta,      'Cross-dataset delta (MSR-VTT minus MSVD).','tab:delta'),
]
for fname, df, cap, lbl in table_defs:
    ltx = to_latex(df, cap, lbl)
    p   = os.path.join(TABLES_DIR, f'{fname}.tex')
    with open(p,'w') as f: f.write(ltx)
    print(f'Saved {os.path.basename(p)}')
if not df_timing.empty:
    mean_cols = [c for c in df_timing.columns if 'mean' in c]
    ltx = to_latex(df_timing[mean_cols].round(3),
                   'Mean inference latency (s/video).','tab:latency',highlight_best=False)
    with open(os.path.join(TABLES_DIR,'tab_latency.tex'),'w') as f: f.write(ltx)
    print('Saved tab_latency.tex')

In [ ]:
# Master 3-row combined figure
fig = plt.figure(figsize=(20,18))
gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.52,wspace=0.38)

# Row 1: main results
for col_j, ds in enumerate(['MSVD','MSR-VTT']):
    ax = fig.add_subplot(gs[0, col_j*1:(col_j*1)+2] if ds=='MSVD' else gs[0,2])
    df_plot = dfs[ds]; mdl_p = list(df_plot.index)
    x = np.arange(len(METRICS)); n=len(mdl_p); w=0.10
    for i,(model,color) in enumerate(zip(mdl_p,PALETTE)):
        vals=[df_plot.loc[model,m] for m in METRICS]
        off=(i-n/2+0.5)*w
        ax.bar(x+off,vals,w,label=model,color=color,alpha=0.88,edgecolor='white',linewidth=0.4)
    ax.set_title(f'{ds} Results',fontsize=11)
    ax.set_xticks(x); ax.set_xticklabels(METRICS,fontsize=8)
    ax.set_ylabel('Score',fontsize=9); ax.legend(fontsize=6,ncol=2)
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.grid(True,alpha=0.3,linestyle='--'); ax.set_axisbelow(True)

# Row 2: heatmap + violin
ax_h = fig.add_subplot(gs[1,:2])
im = ax_h.imshow(data_h,cmap='RdYlGn',aspect='auto',vmin=-0.15,vmax=0.15)
plt.colorbar(im,ax=ax_h,label='Delta')
ax_h.set_xticks(range(len(METRICS))); ax_h.set_xticklabels(METRICS,fontsize=9)
ax_h.set_yticks(range(len(df_delta))); ax_h.set_yticklabels(df_delta.index,fontsize=8)
ax_h.set_title('Generalisation Heatmap',fontsize=11)
for i in range(len(df_delta)):
    for j in range(len(METRICS)):
        ax_h.text(j,i,f'{data_h[i,j]:+.3f}',ha='center',va='center',fontsize=7.5)
ax_v = fig.add_subplot(gs[1,2])
if pv_scores['MSVD']:
    mdls_v=[m for m in MODEL_LABELS.values() if m in pv_scores['MSVD']]
    data_v=[pv_scores['MSVD'][m]['CIDEr'].values.astype(float) for m in mdls_v]
    parts=ax_v.violinplot(data_v,positions=range(len(mdls_v)),showmedians=True,showextrema=True)
    for pc,color in zip(parts['bodies'],PALETTE): pc.set_facecolor(color);pc.set_alpha(0.7)
    ax_v.set_xticks(range(len(mdls_v)))
    ax_v.set_xticklabels(mdls_v,rotation=30,ha='right',fontsize=7)
    ax_v.set_ylabel('Per-video CIDEr',fontsize=9)
    ax_v.set_title('CIDEr Distribution (MSVD)',fontsize=11)
    ax_v.spines[['top','right']].set_visible(False)
    ax_v.yaxis.grid(True,alpha=0.3,linestyle='--'); ax_v.set_axisbelow(True)

# Row 3: frame ablation + latency
ax_fa = fig.add_subplot(gs[2,:2])
for model,color in zip(abl_models,PALETTE):
    for ds,ls in [('MSVD','-'),('MSR-VTT','--')]:
        if ds not in abl_frame.get(model,{}): continue
        vals=[abl_frame[model][ds].get(f'{n} frames',{}).get('scores',{}).get('CIDEr',0)
              for n in FRAME_COUNTS]
        ax_fa.plot(FRAME_COUNTS,vals,ls+'o',label=f'{model}/{ds}',
                   color=color,linewidth=1.8,markersize=6)
ax_fa.set_title('Frame-Count Ablation — CIDEr',fontsize=11)
ax_fa.set_xticks(FRAME_COUNTS); ax_fa.set_xlabel('Frames'); ax_fa.set_ylabel('CIDEr')
ax_fa.legend(fontsize=6,ncol=2)
ax_fa.spines[['top','right']].set_visible(False)
ax_fa.yaxis.grid(True,alpha=0.3,linestyle='--'); ax_fa.set_axisbelow(True)

ax_lt = fig.add_subplot(gs[2,2])
if not df_timing.empty:
    mean_c=[c for c in df_timing.columns if 'mean' in c]
    xp2=np.arange(len(df_timing)); w2=0.35
    for j,(col,ds) in enumerate(zip(mean_c,DATASETS)):
        color='#4C72B0' if ds=='MSVD' else '#C44E52'
        ax_lt.bar(xp2+(j-0.5)*w2,df_timing[col].values,w2,label=ds,color=color,alpha=0.85)
    ax_lt.set_xticks(xp2)
    ax_lt.set_xticklabels(df_timing.index,rotation=30,ha='right',fontsize=7)
    ax_lt.set_ylabel('s/video',fontsize=9); ax_lt.set_title('Latency',fontsize=11)
    ax_lt.legend(fontsize=8)
    ax_lt.spines[['top','right']].set_visible(False)
    ax_lt.yaxis.grid(True,alpha=0.3,linestyle='--'); ax_lt.set_axisbelow(True)

fig.suptitle('Video Captioning — Complete Analysis: MSVD & MSR-VTT',fontsize=15,y=1.005)
plt.savefig(os.path.join(FIGS_DIR,'fig_master.pdf'),dpi=300,bbox_inches='tight')
plt.savefig(os.path.join(FIGS_DIR,'fig_master.png'),dpi=200,bbox_inches='tight')
plt.show()
print('Master figure saved.')

In [ ]:
import glob
print('\n' + '='*60)
print('  Notebook 05 — All Outputs')
print('='*60)
cats = [('Figures (PDF)', os.path.join(FIGS_DIR,'*.pdf')),
        ('Figures (PNG)', os.path.join(FIGS_DIR,'*.png')),
        ('LaTeX tables',  os.path.join(TABLES_DIR,'*.tex')),
        ('Summary CSVs',  os.path.join(RESULTS_DIR,'*.csv')),
        ('Ablation JSONs',os.path.join(RESULTS_DIR,'*ablation*.json'))]
for label, pattern in cats:
    files = sorted(glob.glob(pattern))
    print(f'\n  {label} ({len(files)}):')
    for fp in files:
        print(f'    {os.path.basename(fp):<52} {os.path.getsize(fp):>8,} B')